# phases 1 & 2 : Exploration et Nettoyage

Ce notebook sera dédié à :
 - la phase **exploration** pour comprendre mieux la structure des données

 - et  **le nettoyage** pour rendre les données fiables pour l'analyse


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#e_sales = pd.read_csv("../datasets/raw/Online Retail Data Set.csv") 
# provoque une erreur à cause du nom de fichier que python ne comprend pas

dossier = Path(r"C:\Users\HP\Documents\Projets de competences\Data\data-online-retail-month2\datasets\raw")

for fichier in dossier.glob("*.csv"):
    nouveau_nom = fichier.stem.lower().replace(" ", "_")
    nouveau_chemin = fichier.parent / f"{nouveau_nom}{fichier.suffix.lower()}"
    
    if fichier != nouveau_chemin:
        fichier.rename(nouveau_chemin)
        print(f"Renommé : {fichier.name} → {nouveau_chemin.name}")

In [4]:
e_sales = pd.read_csv("../datasets/raw/online_retail_data_set.csv", encoding="latin-1")
e_sales.to_csv('../datasets/raw/online_retail_data_set_utf8.csv', encoding='utf-8', index=False)
e_sales.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [5]:
print('taille du dataset: ',e_sales.shape,'\n')
print("forme du dataset: \n")
e_sales.info()

taille du dataset:  (541909, 8) 

forme du dataset: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


---
**constats**

- le dataset a une taille de 541909 lignes et 8 colonnes
- les informations sur les colonnes nous apprend qu'il y'a des valeurs 
  manquantes sur description et customerID

In [11]:
e_sales.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


---
**constats**

D'abord, je ne pense pas considérer CustomerID car je ne saurais pas interpréter les données 
que cette colonne présente, elle a été placée comme un float, mais je la considère du même niveau que InvoiceNo
C'est-à-dire des ID.

---
- il n'y a pas de valeur manquante sur les quantités de produits ainsi que leurs prix unitaires
- pour les colonnes quantity et unitprice, les différences entre la moyenne et l'écart-type sont importantes :
  
                - 218.08 std pour une quantité moyenne de 9.55
  
                - 96.75 std pour un prix unique moyen de 4.6 £
  
  pour l'un ou l'autre, les quantités et les prix varient très fortement autour des moyennes.
  Il faut donc se référer à la médiane pour avoir des interprétations plus justes

  Cette dispersion suggère que certaines transactions présentent des volumes ou des prix
  
  très éloignés du cas courant. À investiguer en phase de nettoyage.
---
- Quantity et UnitPrice présentent des valeurs négatives (min : -80995 et -11062)
  Valeurs aberrantes ou retours/annulations ? À investiguer en phase nettoyage
- Distribution des lignes de transaction :
    - 25% des lignes portent sur 1 article ou moins à moins de 1.25 £
    - 50% des lignes portent sur 3 articles ou moins à moins de 2.08 £
    - 75% des lignes portent sur 10 articles ou moins à moins de 4.13 £

In [12]:
e_sales.describe(include = "all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,31-10-2011 14:41,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.690570,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1713.600303,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16791.000000,NaN


**constats**

- Certaines lignes ont un StockCode sans Description associée (540455 lignes   pour 541909 descriptions). Par ailleurs, on compte 4223 descriptions uniques pour seulement 
  4070 StockCodes uniques — un même code produit peut donc avoir plusieurs descriptions 
  différentes. À noter pour la phase nettoyage.

- Le produit le plus fréquent est WHITE HANGING HEART T-LIGHT HOLDER (2369 lignes).
  Le pays le plus représenté est United Kingdom (495478 lignes sur 541909).
  United Kingdom représente donc environ 91% des transactions du dataset.

In [9]:
# détecter  les doublons s'il y en a:

print('Doublons totaux :', e_sales.duplicated().sum(),'\n') 

 
# Détecter les doublons sur la colonne stockcode
print('Doublons sur StockCode :', e_sales.duplicated('StockCode').sum(),"\n") 

# afficher les lignes  entièrement dupliquées 
print('doublons: \n')
print(e_sales.duplicated().head(15),'\n') 


# afficher une combinaison de colonnes pour trouver trouver des lignes identiques si possible
print('possibles lignes identiques: \n')
print(e_sales[e_sales.duplicated(["InvoiceNo", "StockCode", 'Description','Quantity','UnitPrice'])].head(30))

Doublons totaux : 5268 

Doublons sur StockCode : 537839 

doublons: 

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
dtype: bool 

possibles lignes identiques: 

     InvoiceNo StockCode                          Description  Quantity  \
517     536409     21866          UNION JACK FLAG LUGGAGE TAG         1   
527     536409     22866        HAND WARMER SCOTTY DOG DESIGN         1   
537     536409     22900      SET 2 TEA TOWELS I LOVE LONDON          1   
539     536409     22111         SCOTTIE DOG HOT WATER BOTTLE         1   
555     536412     22327    ROUND SNACK BOXES SET OF 4 SKULLS         1   
587     536412     22273                 FELTCRAFT DOLL MOLLY         1   
589     536412     22749    FELTCRAFT PRINCESS CHARLOTTE DOLL         1   
594     536412     22141       CHRISTMAS CRAFT TREE TOP ANGEL         1   
598     536412     21448     

---
**Constats:**

Dans tout le dataset, il y a au total **5268 doublons.**

---

Les colonnes à regarder pour un dataset de ventes pour voir où se situe les potentiels occurrences

sont celles où on doit avoir le plus de valeurs distinctes possibles

or , StockCode qui joue le rôle d'identifiant clé produit, revient plusieurs fois sur plusieurs valeurs

un client peut commander plusieurs fois le même produit dans le même intervalle de temps 

depuis le même pays, ça n'a rien d'anormal.

donc il fallait regarder également le numéro de transaction, combiné à l'identifiant produit

et également inclure plusieurs autres critères pour être sûr de trouver des lignes à l'identique

mais le volume, étant important, je me suis arrêté aux 30 premières lignes , suffisantes pour trouver 

2 lignes identiques aux numéros de ligne 601 et 604

In [10]:
# suppression des doublons

e_sales_not_duplicated = e_sales.drop_duplicates() 

# pour vérifier combien de lignes ont été supprimées
print("Avant :", e_sales.shape,'\n')
print("Après :", e_sales_not_duplicated.shape,'\n')
print("Lignes supprimées :", len(e_sales) - len(e_sales_not_duplicated),'\n')


Avant : (541909, 8) 

Après : (536641, 8) 

Lignes supprimées : 5268 



In [11]:
# Diagnostic des valeurs manquantes

# Nombre de valeurs manquantes par colonne 
print("valeurs manquantes par colonnes: \n")
print(f"{(e_sales_not_duplicated.isnull().sum())}\n") 


# Valeurs distincte seule colonne 
e_sales_not_duplicated["Description"].unique()

e_sales_not_duplicated["Quantity"].unique()

e_sales_not_duplicated["CustomerID"].unique() 


# Description: on garde les valeurs qui sont cohérentes et on élimine
# les NAN ainsi que les valeurs erronées

colonne = e_sales_not_duplicated["Description"]
conditions = (
    colonne.str.isupper() |          # tout en majuscules
    colonne.str.startswith('*') |    # commence par *
     (
        colonne.str.split().str[0].str.isdigit() &   # premier mot = nombre
        colonne.str.split().str[1].str.isupper()     # deuxième mot en majuscules
    ) |
    (colonne == "Manual")            # égal à Manual
)

e_sales_description_clean = e_sales_not_duplicated[conditions]

liste_noire = ['AMAZON', 
               'DAMAGED', 
               'CHECK', 
               'WET/MOULDY', 
               'POSSIBLE DAMAGES OR LOST?',
               'MERCHANT CHANDLER CREDIT ERROR, STO',
               'MIA',
               'FOUND',
              ]
e_sales_description_clean = e_sales_description_clean[~e_sales_description_clean["Description"].isin(liste_noire)]

print("noms de produits valides: \n")
print(e_sales_description_clean)
print("\n")

# Comparer avant / après 
print('Lignes perdues pour description:', len(e_sales_not_duplicated) - len(e_sales_description_clean)) 
print('\n')


# pour les valeurs NAN de customerID, étant donné qu'il représente 25% 
# du dataset total, vaut mieux les garder dans le dataset au lieu de les retirer
# et les CustomerID manquants sont associés à des quantités positives et des prix unitaires positifs également
# donc ce sont sûrement des achats anonymes

e_sales_description_clean = e_sales_description_clean.copy(deep = True)

e_sales_description_clean['CustomerID'] = e_sales_description_clean['CustomerID'].fillna('Unknown') 

e_sales_description_clean['CustomerID'] = e_sales_description_clean["CustomerID"].astype(str)

print(e_sales_description_clean[e_sales_description_clean['CustomerID'] == 'Unknown'],'\n')

print("nouvelles valeurs manquantes par colonnes: \n")
print(f"{(e_sales_description_clean.isnull().sum())}\n")

valeurs manquantes par colonnes: 

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64

noms de produits valides: 

       InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DO

---
**Récapitulatif**


**Description** et **CustomerID** sont des colonnes dont pandas a détecté des NAN avec isnull().sum()

---
**Description** a dû être nettoyé en premier car il présentait non seulement des NAN 

mais aussi des valeurs bizarres qui n'avaient rien à voir avec des noms de produits.

Les lignes associés aux autres colonnes ne présentaient pas de valeur métier qui justifait de les garder

   - unitPrice à zéro
   - Quantity parfois positive , et parfois négative qui n'avait rien à voir avec des annulations ou des retours
   - des CustomerID manquant

il fallait donc supprimer ces lignes, mais étant donné qu'il y avait aussi des valeurs aberrantes à prendre en compte

dropna() aurait été insuffisant, il ne fallait garder que des lignes qui correspondait bien à des noms de produits

noms valides uniquement en majuscules, ou commençant par un chiffre suivi d'une majuscule, la valeur Manual étant une exception,

il fallait l'inclure dans le filtre appliqué par condition, on crée donc un nouveau DataFrame contenant uniquement les lignes valides.

et on a établi une liste noire qui contient des valeurs aberrantes , qui, bien qu'en majuscules, ne correspondent pas à des noms de produits.

---
Pour **CustomerID**, la décision a été de modifier la colonne au lieu de supprimer les NAN présents

car non seulement la taille de ces NAN représentait 25% environ du dataset, mais elles apportaient

de la valeur métier, les colonnes associés présentant des données correctes, il a été décidé d'utiliser fillna()

Pour signifier des achats anonymes.

Comme une colonne devait être modifiée, il a fallu créer une copie complète du dataframe où les lignes de **Description**

avaient déjà été traités, avec la méthode copy(deep=True)

et bien sûr de changer le type de la colonne **CustomerID** qui a été identifiée comme un float au départ.

In [12]:
# colonne correspondant aux prix totaux
e_sales_description_clean['TotalPrice'] = e_sales_description_clean['Quantity']* e_sales_description_clean['UnitPrice']
e_sales_description_clean['TotalPrice'] = e_sales_description_clean['TotalPrice'].round(2)
print(e_sales_description_clean.head())

# Correction du type de colonne sur Invoice Date

print(e_sales_description_clean.dtypes,"\n")

# on convertit le type object de Invoice date en type Datetime
e_sales_description_clean['InvoiceDate'] = pd.to_datetime(e_sales_description_clean['InvoiceDate'], format='%d-%m-%Y %H:%M')
print(e_sales_description_clean.dtypes,"\n")


# Extraire des informations temporelles
e_sales_description_clean['year'] = e_sales_description_clean['InvoiceDate'].dt.year
e_sales_description_clean['month'] = e_sales_description_clean['InvoiceDate'].dt.month
e_sales_description_clean['day'] = e_sales_description_clean['InvoiceDate'].dt.day_name()

print("\n")
print(e_sales_description_clean[['InvoiceDate','year','month','day']].head())



  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

        InvoiceDate  UnitPrice CustomerID         Country  TotalPrice  
0  01-12-2010 08:26       2.55    17850.0  United Kingdom       15.30  
1  01-12-2010 08:26       3.39    17850.0  United Kingdom       20.34  
2  01-12-2010 08:26       2.75    17850.0  United Kingdom       22.00  
3  01-12-2010 08:26       3.39    17850.0  United Kingdom       20.34  
4  01-12-2010 08:26       3.39    17850.0  United Kingdom       20.34  
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float

In [20]:
zero_price = e_sales_description_clean[e_sales_description_clean['UnitPrice'] == 0]

print(zero_price['Description'].value_counts().head(10),'\n')
print(zero_price['InvoiceNo'].str.startswith('C').sum(), "annulations \n")
print((zero_price['CustomerID'] == 'Unknown').sum(), "clients inconnus \n")

# Isolation des transactions à prix nul pour référence
df_zero_price = e_sales_description_clean[e_sales_description_clean['UnitPrice'] == 0]
print(f"{len(df_zero_price)} transactions à prix nul isolées")

# Dans l'analyse sur le CA, on les exclura explicitement au moment du calcul

Description
FRENCH BLUE METAL DOOR SIGN 1      9
FRENCH BLUE METAL DOOR SIGN 8      8
OWL DOORSTOP                       7
RECIPE BOX PANTRY YELLOW DESIGN    7
FRENCH BLUE METAL DOOR SIGN 3      7
FRENCH BLUE METAL DOOR SIGN 4      7
FRENCH BLUE METAL DOOR SIGN 5      6
FRENCH BLUE METAL DOOR SIGN 6      6
RED KITCHEN SCALES                 6
FRENCH BLUE METAL DOOR SIGN 7      6
Name: count, dtype: int64 

0 annulations 

361 clients inconnus 

401 transactions à prix nul isolées


---
### Investigation de UnitPrice
---
Le diagnostic initial s'était concentré sur les colonnes où Pandas avait détecté

des NaN explicites : **Description** et **CustomerID**.

---
En nettoyant **Description**, une partie des lignes à **UnitPrice** = 0 a été supprimée

indirectement — elles portaient des valeurs incohérentes qui ne correspondaient

pas à des noms de produits. En traitant **CustomerID**, les NaN restants ont été

conservés via fillna('Unknown') car ils représentaient 25 % du dataset

et correspondaient à des transactions apparemment valides.

---

Ce n'est qu'en phase de vérification post-nettoyage qu'il est apparu que

cette décision avait préservé **401** lignes à **UnitPrice = 0** — soit les transactions

à prix nul portant des descriptions et des quantités valides.

---

L'investigation révèle que ces lignes ne sont pas des annulations **(0 InvoiceNo préfixé C)**, 

Mais elles sont majoritairement associées à des clients anonymes (361/401).

Leur nature exacte reste indéterminée : soldes, échantillons, promotions,saisies incomplètes

le dataset ne permet pas de trancher.

---

Face à cette incertitude, la suppression aurait été une décision arbitraire.

**Ces 401 lignes (0,075 % du dataset) sont donc conservées**, mais leur **TotalPrice** étant nul, 

elles seront explicitement exclues des calculs de chiffre d'affaires au moment de l'analyse. 

L'incertitude est donc documentée, pas masquée.

In [13]:
df_clean = e_sales_description_clean

df_clean.to_csv("../datasets/processed/online_retail_clean.csv", index=False)
print("Nouvelle taille:", df_clean.shape, "\n")
df_clean.info()

Nouvelle taille: (532663, 12) 

<class 'pandas.core.frame.DataFrame'>
Index: 532663 entries, 0 to 541908
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    532663 non-null  object        
 1   StockCode    532663 non-null  object        
 2   Description  532663 non-null  object        
 3   Quantity     532663 non-null  int64         
 4   InvoiceDate  532663 non-null  datetime64[ns]
 5   UnitPrice    532663 non-null  float64       
 6   CustomerID   532663 non-null  object        
 7   Country      532663 non-null  object        
 8   TotalPrice   532663 non-null  float64       
 9   year         532663 non-null  int32         
 10  month        532663 non-null  int32         
 11  day          532663 non-null  object        
dtypes: datetime64[ns](1), float64(2), int32(2), int64(1), object(6)
memory usage: 48.8+ MB


---
**Bilan – Phases d’Exploration et de Nettoyage des Données**

---
  **Phase d’Exploration (Data Exploration)**

---
Objectifs 

L’objectif principal de cette phase était de : comprendre la structure globale du dataset

---
1 - Analyse du dataset
Le dataset contient 541 909 lignes et 8 colonnes

Les variables principales décrivent des transactions commerciales (`Description`, `Quantity`, `UnitPrice`, `InvoiceDate`, `CustomerID`, `Country`)

---
2 - Types de données :

`CustomerID` interprété comme float donc incohérent pour un identifiant

---
3 - Distribution des variables :

- Forte dispersion sur `Quantity` et `UnitPrice`
- Moyennes peu représentatives = utilisation recommandée de la médiane
- Analyse des variables catégorielles
    Incohérence entre :
                      -`StockCode` (4070 uniques)
                      -`Description` (4223 uniques)
  
    Une même clé produit possède plusieurs descriptions = problème de standardisation possible
  
- Produit le plus fréquent : WHITE HANGING HEART T-LIGHT HOLDER
- Pays dominant: United Kingdom (~91% des transactions) : Dataset fortement déséquilibré géographiquement
- Analyse multi-colonnes : `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `UnitPrice`

Conclusion : certains doublons sont des erreurs = suppression nécessaire

---
  **Phase de Nettoyage (Data Cleaning)**

---

Objectifs:

---
    - Identifier les anomalies, valeurs manquantes et incohérences

    - Gérer les anomalies et valeurs non exploitables
    
    - Préparer le dataset pour l’analyse ou la visualisation
---
1-  Traitement des doublons:

---
 - Détection des doublons: 5268 lignes dupliquées détecté
 - Suppression des doublons avec drop_duplicates()
 - Vérification du nombre de lignes supprimées
 - Résultat : dataset allégé sans perte d’information pertinente

---
2 - Gestion des valeurs manquantes et incohérentes

---

Colonne `Description`
Problèmes identifiés :

                     - Valeurs nulles : 1454
                     - Valeurs aberrantes (symboles, formats incohérents)
                     
Stratégie :Filtrage basé sur des règles métier :

          - Texte en majuscules
          - Formats cohérents avec des noms de produits
          - Suppression des lignes non conformes
          
Colonne `CustomerID`

Présence de valeurs nulles : considérées comme non exploitable dans l’état actuel

Les valeurs NaN de `CustomerID` associées à des lignes `Description` aberrantes ont été supprimées indirectement. 

Les NaN restants ont été remplacés par 'Unknown' car ils correspondaient à des achats anonymes valides (~25% du dataset).

---
3 - Suppression des valeurs aberrantes

---

`Quantity` négative hors annulation (`InvoiceNo` sans préfixe C) :

supprimées indirectement via le filtre Description — ces lignes

portaient systématiquement des valeurs non conformes.

Résultat vérifié post-nettoyage : 0 ligne restante.

`UnitPrice` = 0 :

401 lignes identifiées en vérification post-nettoyage.

Nature indéterminée (soldes, échantillons, saisies incomplètes) —

le dataset ne permet pas de trancher. Conservées dans le dataset,

elles seront exclues des calculs de CA au moment de l'analyse.

Poids : 0,075 % du dataset.

---
4 - Feature Engineering

--- 
Création de nouvelles variables : - `TotalPrice` = `Quantity` × `UnitPrice`
                                 Arrondi à 2 décimales pour cohérence financière
                                 
                                  - Transformation des dates : Conversion de `InvoiceDate` en format datetime
                                  
                                  - Extraction de variables temporelles : Année (`year`), Mois (`month`), Jour(`Day`)
---
5 -  Export du dataset nettoyé

---

Sauvegarde du dataset final :online_retail_clean.csv

Nombre de lignes finales: 532663 ; Nombre de colonnes : 12

Vérification finale :
  - Structure cohérente
  - Types de données corrects

**Ce dataset est prêt pour les analyses**

In [17]:
df_clean[df_clean['Description'].str.contains(',', na=False)]['Description'].unique()

array(['AIRLINE LOUNGE,METAL SIGN', 'FANCY FONT BIRTHDAY CARD, ',
       'TRAY, BREAKFAST IN BED', 'SWISS ROLL TOWEL, CHOCOLATE  SPOTS',
       'BIRTHDAY CARD, RETRO SPOT', 'ACRYLIC JEWEL ICICLE, PINK',
       'TUMBLER, BAROQUE', 'TUMBLER, NEW ENGLAND',
       'METAL SIGN,CUPCAKE SINGLE HOOK',
       'SET 3 RETROSPOT TEA,COFFEE,SUGAR', 'ELEPHANT, BIRTHDAY CARD, ',
       'HOOK, 1 HANGER ,MAGIC GARDEN', 'FEATHER PEN,COAL BLACK',
       'FEATHER PEN,LIGHT PINK', 'FEATHER PEN,HOT PINK',
       'ART LIGHTS,FUNK MONKEY', 'NURSERY A,B,C PAINTED LETTERS',
       'CHRISTMAS GARLAND STARS,TREES',
       'LARGE CAKE TOWEL, CHOCOLATE SPOTS', 'KEY FOB , GARAGE DESIGN',
       'DIAMANTE HEART SHAPED WALL MIRROR, ',
       'CAKESTAND, 3 TIER, LOVEHEART', 'DECORATION HEN ON NEST, HANGING',
       'BLACK TEA,COFFEE,SUGAR JARS', 'KEY FOB , FRONT  DOOR ',
       'KEY FOB , BACK DOOR ', 'WRAP, BILLBOARD FONTS DESIGN',
       'KEY FOB , SHED', 'DECOUPAGE,GREETING CARD,',
       'SWISS ROLL TOWEL, PINK  SP

In [18]:
import csv
df_clean.to_csv('C:/Users/HP/Documents/Projets de competences/Data/data-online-retail-month2/datasets/processed/online_retail_clean.csv', 
          index=False, 
          encoding='utf-8',
          quoting=csv.QUOTE_ALL)